In [1]:
import pandas as pd

# ==========================================
# 1. 配置文件路径
# ==========================================
# 刚才生成的图像标签清单（作为你要保留的患者白名单）
IMAGE_COHORT_PATH = '../data/pneumonia_labeled.csv' 
# 你的包含 44 万行、100 多列的 ED visit 主数据
MASTER_DATA_PATH = '../data/mimic_all.csv'     
# 最终输出的结构化特征文件
OUTPUT_STRUCTURED_PATH = '../data/structured_clinical_features.csv'

print("正在加载患者白名单...")
# 加载刚刚确定的 5204 个患者的 cohort
image_df = pd.read_csv(IMAGE_COHORT_PATH)
# 提取我们需要保留的 subject_id 列表
target_subjects = set(image_df['subject_id'].unique())
print(f"目标图像队列中共有 {len(target_subjects)} 名独立患者。")

# ==========================================
# 2. 加载并过滤 Master Data
# ==========================================
print("\n正在处理 44 万行 Master Data...")

master_df = pd.read_csv(MASTER_DATA_PATH)
print(f"原始主数据大小: {master_df.shape[0]} 行, {master_df.shape[1]} 列")

# 过滤：只保留存在于白名单中的 subject_id
filtered_master = master_df[master_df['subject_id'].isin(target_subjects)]

# ==========================================
# 3. 处理一次就诊 (Visit) 的唯一性
# ==========================================
# 因为 44 万行通常包含同一个患者的多次就诊记录，我们需要去重
# 这里的逻辑是：为了匹配我们之前提取的“第一张 X 光片”，
# 我们也提取该患者在急诊的“第一次”就诊记录，或者直接基于 subject_id 去重保留第一条
filtered_master = filtered_master.sort_values(by=['subject_id'])
unique_patient_features = filtered_master.drop_duplicates(subset=['subject_id'], keep='first')

print(f"过滤并去重后，提取出 {len(unique_patient_features)} 名患者的临床特征。")

正在加载患者白名单...
目标图像队列中共有 5204 名独立患者。

正在处理 44 万行 Master Data...
原始主数据大小: 448804 行, 168 列
过滤并去重后，提取出 5204 名患者的临床特征。


In [2]:
# 假设你上一步生成的 DataFrame 叫 unique_patient_features
# 1. 直接打印所有列名
print("All Column Names in Master Data:")
print(unique_patient_features.columns.tolist())

# 2. 生成一个简单的特征质量报告 (保存为 CSV 方便在 Excel 查看)
# 统计每列的缺失值比例
quality_report = pd.DataFrame({
    'column_name': unique_patient_features.columns,
    'missing_rate': unique_patient_features.isnull().mean() * 100,
    'unique_values': unique_patient_features.nunique()
})

# 按缺失率从低到高排序
quality_report = quality_report.sort_values(by='missing_rate')

All Column Names in Master Data:
['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'anchor_age', 'gender', 'anchor_year', 'dod', 'admittime', 'dischtime', 'deathtime', 'ethnicity', 'edregtime', 'edouttime', 'insurance', 'in_year', 'age', 'outcome_inhospital_mortality', 'ed_death', 'before_ed_mortality', 'ed_los', 'intime_icu', 'time_to_icu_transfer', 'outcome_icu_transfer_12h', 'outcome_hospitalization', 'outcome_critical', 'n_ed_30d', 'n_ed_90d', 'n_ed_365d', 'next_ed_visit_time', 'next_ed_visit_time_diff', 'outcome_ed_revisit_3d', 'n_hosp_30d', 'n_hosp_90d', 'n_hosp_365d', 'n_icu_30d', 'n_icu_90d', 'n_icu_365d', 'ed_los_hours', 'time_to_icu_transfer_hours', 'next_ed_visit_time_diff_days', 'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_pain', 'triage_acuity', 'chiefcomplaint', 'chiefcom_chest_pain', 'chiefcom_abdominal_pain', 'chiefcom_headache', 'chiefcom_shortness_of_breath', 'chiefcom_back_pain', 'chiefcom_co

In [3]:
import pandas as pd

# 1. 加载数据
image_df = pd.read_csv('../data/pneumonia_labeled.csv')
master_df = pd.read_csv('../data/mimic_all.csv')

print(f"合并前图像数量: {len(image_df)}")

# 2. 处理时间格式 (这是最关键的一步)
# 确保 master_df 中的入科和出科时间是 datetime 格式
master_df['intime'] = pd.to_datetime(master_df['intime'])
master_df['outtime'] = pd.to_datetime(master_df['outtime'])

# 处理 X 光片的时间
# MIMIC-CXR 的 StudyDate 格式通常是 YYYYMMDD 的整数或字符串
image_df['StudyDate'] = pd.to_datetime(image_df['StudyDate'].astype(str), format='%Y%m%d', errors='coerce')

# 3. 基于 subject_id 进行笛卡尔积合并 (把该患者所有的急诊记录和所有的X光片先拼起来)
# 注意：这会产生多行，没关系，我们下一步会过滤
merged_df = pd.merge(image_df, master_df, on='subject_id', how='inner')

# 4. 时间过滤：只保留 X光片拍摄日期 在 急诊就诊期间 的记录
# 考虑到急诊跨夜的情况，我们比较 X光片的日期 是否在 intime 的日期和 outtime 的日期之间
# 使用 dt.date 忽略具体的小时分钟，只比较天，这样更稳健
merged_df['intime_date'] = merged_df['intime'].dt.date
merged_df['outtime_date'] = merged_df['outtime'].dt.date
merged_df['cxr_date'] = merged_df['StudyDate'].dt.date

# 核心过滤逻辑：拍片日期 >= 入院日期 且 拍片日期 <= 出院日期
aligned_df = merged_df[
    (merged_df['cxr_date'] >= merged_df['intime_date']) & 
    (merged_df['cxr_date'] <= merged_df['outtime_date'])
]

# 5. 去重：如果一次完整的急诊 (stay_id) 里拍了多张满足条件的片子，保留第一张
final_aligned_df = aligned_df.drop_duplicates(subset=['subject_id', 'stay_id'], keep='first')

print(f"时间对齐后，成功匹配的就诊记录数: {len(final_aligned_df)}")



合并前图像数量: 5204
时间对齐后，成功匹配的就诊记录数: 2646


In [ ]:
# 6. 现在你可以放心地保存并使用这批数据了
# 这时候，这一行数据里的 Triage 信息，就绝对是患者拍这张片子那次来急诊时的真实体征！
final_aligned_df.to_csv('../data/structured_clinical_features_time_aligned.csv', index=False)